# Part 2: Connect Data and Create Visualizations (Tableau)
Each step: the Tableau clicks first, then a cell that prints the numbers Tableau should show. If Tableau matches, the step is done.

Run `01_data_preparation.ipynb` first.

In [1]:
# Load the clean file from Part 1
from pathlib import Path
import pandas as pd

df = pd.read_csv("evaluation_data_clean.csv")
print("PASS clean file loaded:", len(df), "rows")

PASS clean file loaded: 4489 rows


## Step 4: Connect data to Tableau
1. Open Tableau Desktop. Under **Connect > To a File**, click **Text file**.
2. Select `evaluation_data_clean.csv` (path printed at the end of notebook 01) and click **Open**.
3. On the Data Source page check the type icons: `score` **#**, `category` and `model_version` **Abc**, `date` **calendar**.
4. Click **Sheet 1**. In the Data pane, if `evaluation_id` sits under Measures (green), right-click it > **Convert to Dimension**.
5. `score` must be a Measure (green pill). `date` must be a date dimension.

In [2]:
# What Tableau should show after connecting
print("Rows in Tableau data preview:", len(df))
print("Columns:", list(df.columns))
print("Distinct categories:", df["category"].nunique(), "| distinct models:", df["model_version"].nunique())

Rows in Tableau data preview: 4489
Columns: ['evaluation_id', 'category', 'score', 'model_version', 'date']
Distinct categories: 5 | distinct models: 4


## Step 5: Score distribution histogram
1. In the Data pane, right-click `score` > **Create > Bins**. Size of bins: **10**. OK. A new field `score (bin)` appears.
2. Drag `score (bin)` to **Columns**.
3. Drag the count field (`evaluation_data_clean.csv (Count)`, older versions: `Number of Records`) to **Rows**. Tableau draws bars.
4. Title: double-click the sheet title > **Score Distribution**. Rename the sheet tab: **Score Distribution**.

In [3]:
# Reference: evaluations per bin of 10 (Tableau labels each bin by its lower edge)
bins = (df["score"] // 10 * 10).astype(int)
hist = bins.value_counts().sort_index()
print(hist.to_string())
print("\nNote: a score of exactly 100 gets its own bin 100 in Tableau too:", (df["score"] == 100).sum(), "rows")

score
0       84
10     216
20     366
30     551
40     656
50     655
60     642
70     588
80     490
90     219
100     22

Note: a score of exactly 100 gets its own bin 100 in Tableau too: 22 rows


## Step 6: Score distribution by category (box plot)
1. New worksheet. Drag `category` to **Columns**, `score` to **Rows**.
2. Drag `evaluation_id` to **Detail** on the Marks card (one dot per evaluation).
3. **Show Me** > **box-and-whisker plot**. Set the mark type to **Circle**.
4. Title: **Score Distribution by Category**. Rename the sheet: **Scores by Category**.

In [4]:
# Reference: box plot values per category (median line and box edges)
box = df.groupby("category")["score"].describe()[["min", "25%", "50%", "75%", "max"]].round(1)
box.columns = ["min", "lower quartile", "median", "upper quartile", "max"]
print(box.sort_values("median", ascending=False).to_string())

                        min  lower quartile  median  upper quartile    max
category                                                                  
instruction_following  21.3            60.4    74.5            85.6  100.0
knowledge               6.8            52.6    68.5            80.8  100.0
code                    0.7            35.5    49.5            64.2   97.2
tool_calling            1.7            34.1    47.9            62.1   95.3
reasoning               0.0            25.3    39.2            54.6   96.5


## Step 7: Category comparison chart (Option B: bar chart with averages)
1. New worksheet. Drag `category` to **Rows**, `score` to **Columns**. Right-click the pill > **Measure > Average**.
2. Click **Sort Descending** in the toolbar.
3. Drag `AVG(score)` to **Color** (edit colors > Orange-Blue Diverging) and to **Label**. Format the label number to 1 decimal.
4. Overall average line: right-click the score axis > **Add Reference Line** > Scope **Entire Table**, Value **AVG(score)**, aggregation **Total**, Label **Overall average**.
5. Title: **Category Performance Comparison**. Rename the sheet: **Category Comparison**.

In [5]:
# Reference: average score per category, best first, plus the overall average line
avg = df.groupby("category")["score"].mean().round(1).sort_values(ascending=False)
overall = round(df["score"].mean(), 1)
for cat, value in avg.items():
    print(f"{cat:<22} {value:>5}  {'above' if value > overall else 'below'} overall")
print(f"\nOverall average (reference line, aggregation Total): {overall}")
print(f"If Tableau shows {round(avg.mean(), 1)} instead, the line uses Average of the bars: switch aggregation to Total")

instruction_following   72.2  above overall
knowledge               66.0  above overall
code                    49.6  below overall
tool_calling            48.3  below overall
reasoning               40.8  below overall

Overall average (reference line, aggregation Total): 55.3
If Tableau shows 55.4 instead, the line uses Average of the bars: switch aggregation to Total


## Step 8: Add interactive filters
1. Right-click `model_version` in the Data pane > **Show Filter**. Repeat for `category`.
2. Date: drag `date` to **Filters** > **Range of Dates** > OK, then right-click the pill > **Show Filter**.
3. On each filter card: dropdown arrow > **Apply to Worksheets > All Using This Data Source** (so every chart updates).
4. Filter style: dropdown arrow > **Multiple Values (dropdown)** for model and category.
5. Test: select only **gpt-4** and compare with the numbers below.

In [6]:
# Reference for the filter test: averages per category with only gpt-4 selected
gpt4 = df[df["model_version"] == "gpt-4"]
print("gpt-4 evaluations:", len(gpt4))
print(gpt4.groupby("category")["score"].mean().round(1).sort_values(ascending=False).to_string())

gpt-4 evaluations: 1141
category
instruction_following    72.0
knowledge                65.7
code                     50.5
tool_calling             49.9
reasoning                40.7
